# Women's college basketball intro — sportsdataverse-py

A tour of the NCAA women's basketball (`sdv.wbb`) submodule: teams, rosters, schedules, play-by-play, team stats, standings, conferences, and the parquet data loaders. The wrappers wrap ESPN's women's-college-basketball endpoints and return tidy [polars](https://pola.rs) frames (pass `return_as_pandas=True` for pandas).

R companion: [wehoop](https://wehoop.sportsdataverse.org). Part of the [SportsDataverse](https://py.sportsdataverse.org/docs/ecosystem).

## Setup

```sh
pip install sportsdataverse
```

In [ ]:
import polars as pl
import sportsdataverse as sdv
import sportsdataverse.wbb as wbb

## Teams

`espn_wbb_teams()` returns one wide row per D-I program. Note NCAA team frames carry no conference column — conference membership comes from `espn_wbb_standings()` or `espn_wbb_conferences()` below.

In [ ]:
teams = wbb.espn_wbb_teams()
print(teams.shape)
teams.select(['team_id', 'team_location', 'team_name', 'team_abbreviation', 'team_display_name']).head()

## Team roster

`espn_wbb_team_roster(team_id=..., season=...)` returns one row per player with unprefixed athlete columns. Here is the 2024-25 UConn Huskies (`team_id=2509`), the eventual national champions.

In [ ]:
uconn_roster = wbb.espn_wbb_team_roster(team_id=2509, season=2025)
print(uconn_roster.shape)
uconn_roster.select(
    ['athlete_id', 'full_name', 'jersey', 'position_abbreviation', 'display_height', 'display_weight']
).head(10)

In [ ]:
# South Carolina Gamecocks (team_id=2579), the runners-up
scar_roster = wbb.espn_wbb_team_roster(team_id=2579, season=2025)
scar_roster.select(['athlete_id', 'full_name', 'jersey', 'position_abbreviation']).head()

## Schedule — single date

`espn_wbb_schedule(dates=YYYYMMDD)` returns one row per game. Team-name columns are `home_display_name` / `away_display_name`; `home_score` / `away_score` are **strings**, so cast before arithmetic. April 4, 2025 was the women's Final Four.

In [ ]:
final_four = wbb.espn_wbb_schedule(dates=20250404)
final_four.select(
    ['id', 'date', 'away_display_name', 'away_score', 'home_display_name', 'home_score', 'status_type_completed']
)

## Schedule — date range

Pass a `'YYYYMMDD-YYYYMMDD'` string to span multiple days. Here is the Final Four through the national championship (April 4–6, 2025).

In [ ]:
title_weekend = wbb.espn_wbb_schedule(dates='20250404-20250406')
title_weekend.select(
    ['id', 'date', 'away_display_name', 'away_score', 'home_display_name', 'home_score']
).with_columns(
    pl.col('home_score').cast(pl.Int64, strict=False),
    pl.col('away_score').cast(pl.Int64, strict=False),
)

## Play-by-play

`espn_wbb_pbp(game_id=...)` returns a **dict** of game components (keys like `plays`, `boxscore`, `header`, `winprobability`, ...). The `plays` value is a list of dicts — build a frame with `pl.DataFrame(pbp['plays'], infer_schema_length=None)`. Columns use ESPN dot-notation (`period.number`, `clock.displayValue`, `type.text`, `scoringPlay`).

Game `401746075` is the 2025 national championship: South Carolina vs. UConn.

In [ ]:
pbp = wbb.espn_wbb_pbp(game_id=401746075)
list(pbp.keys())[:10]

In [ ]:
plays = pl.DataFrame(pbp['plays'], infer_schema_length=None)
print(plays.shape)
plays.select(['period.number', 'clock.displayValue', 'type.text', 'scoringPlay', 'text']).head()

In [ ]:
# Scoring plays only, with the running score
plays.filter(pl.col('scoringPlay') == True).select(
    ['period.number', 'clock.displayValue', 'awayScore', 'homeScore', 'text']
).head(8)

## Team season stats

`espn_wbb_team_stats(team_id=..., season=...)` returns a **dict keyed by category** — `{'Averages', 'Totals', 'Misc'}` — each a tidy long frame of `stat_name` / `value` rows. (ESPN's per-player NCAA season stats are usually unavailable, so team stats are the reliable season-level source here.)

In [ ]:
team_stats = wbb.espn_wbb_team_stats(team_id=2509, season=2025)
{k: v.shape for k, v in team_stats.items()}

In [ ]:
# Per-game averages for UConn
team_stats['Averages'].select(['stat_name', 'abbreviation', 'display_value', 'value'])

In [ ]:
team_stats['Totals'].select(['stat_name', 'abbreviation', 'display_value']).head(10)

## Standings

`espn_wbb_standings(season=...)` returns one wide row per team with win/loss records, conference membership, and points-for/against.

In [ ]:
standings = wbb.espn_wbb_standings(season=2025)
print(standings.shape)
standings.select(
    ['team_display_name', 'conference_abbreviation', 'wins', 'losses', 'win_percent', 'points_for', 'points_against']
).sort('win_percent', descending=True).head(10)

## Conferences

`espn_wbb_conferences()` lists the conference groups ESPN tracks, with their group ids — useful for filtering schedules and standings by league.

In [ ]:
conferences = wbb.espn_wbb_conferences()
print(conferences.shape)
conferences.select(['group_id', 'name', 'abbreviation', 'short_name']).head(12)

## Data loaders (parquet releases)

`load_wbb_*(seasons=[...])` read pre-built parquet releases from the [wehoop-wbb-data](https://github.com/sportsdataverse/wehoop-wbb-data) repo and return polars frames — far faster than scraping season-long history through the ESPN endpoints. Loaders include `load_wbb_schedule`, `load_wbb_team_boxscore`, `load_wbb_player_boxscore`, `load_wbb_pbp`, `load_wbb_rosters`, `load_wbb_standings`, and more (`dir(sdv.wbb)` shows the full set).

In [ ]:
schedule_2024 = wbb.load_wbb_schedule(seasons=[2024])
print(schedule_2024.shape)
schedule_2024.select(['id', 'date', 'home_display_name', 'away_display_name']).head()

In [ ]:
team_box_2024 = wbb.load_wbb_team_boxscore(seasons=[2024])
print(team_box_2024.shape)
team_box_2024.select(
    ['game_id', 'team_display_name', 'team_home_away', 'team_score', 'field_goal_pct', 'total_rebounds', 'assists']
).head()

In [ ]:
player_box_2024 = wbb.load_wbb_player_boxscore(seasons=[2024])
print(player_box_2024.shape)
player_box_2024.select(
    ['game_id', 'athlete_display_name', 'team_short_display_name', 'minutes', 'points', 'rebounds', 'assists']
).head()

## Pipeline example: top scorers of the 2023-24 season

Load the season-long player boxscore, then aggregate with polars to find the highest per-game scorers (minimum 20 games played).

In [ ]:
top_scorers = (
    player_box_2024
    .group_by(['athlete_id', 'athlete_display_name', 'team_short_display_name'])
    .agg(
        games=pl.len(),
        total_points=pl.col('points').sum(),
        ppg=pl.col('points').mean().round(1),
    )
    .filter(pl.col('games') >= 20)
    .sort('ppg', descending=True)
    .head(10)
)
top_scorers

## Pipeline example: best scoring offenses

Aggregate the team boxscore to rank programs by average points scored, then join back to the standings to attach each team's record.

In [ ]:
team_offense = (
    team_box_2024
    .group_by(['team_id', 'team_display_name'])
    .agg(
        games=pl.len(),
        ppg=pl.col('team_score').mean().round(1),
    )
    .filter(pl.col('games') >= 20)
    .sort('ppg', descending=True)
    .head(10)
)
team_offense

## Cross-references

- R companion: [wehoop](https://wehoop.sportsdataverse.org)
- Data source: ESPN (women's college basketball)
- Data releases: [wehoop-wbb-data](https://github.com/sportsdataverse/wehoop-wbb-data)
- Plotting: [matplotlib](https://matplotlib.org), [plotnine](https://plotnine.org)

## Where to go next

- API docs: [`docs/docs/wbb/index.md`](../../docs/docs/wbb/index.md)
- Next notebook: [`06_mbb_intro.ipynb`](06_mbb_intro.ipynb) — the parallel men's college basketball surface